# Phase 11 — Performance Report & Dashboard

**Objective:** Compile all results into an Excel dashboard (10 tabs) and a PDF report.

**Dependencies:** All previous phases

**Outputs:**
- `outputs/reports/factor_timing_dashboard.xlsx` — 10-tab Excel dashboard
- `outputs/reports/factor_timing_report.pdf` — PDF summary report

In [1]:
# === Setup & Imports ===
import sys, warnings, logging
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for PDF generation
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path.cwd().parent))

from src.config import (
    PROJECT_ROOT, PROCESSED_DIR, FIGURES_DIR, TABLES_DIR, REPORTS_DIR, LOGS_DIR,
    TICKERS, COLORS
)
from src.validation import validate_parquet
from src.visualization import setup_style, save_fig

setup_style()

PHASE_NUM = 11
logging.basicConfig(
    filename=str(LOGS_DIR / f'phase_{PHASE_NUM}_{datetime.now():%Y%m%d_%H%M}.log'),
    level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s'
)
logger = logging.getLogger(__name__)
logger.info("Phase 11 started")
print("Phase 11 — Performance Report & Dashboard")

Phase 11 — Performance Report & Dashboard


## 11.1 Load All Phase Outputs

In [2]:
# Load all available parquet files
data = {}

parquet_files = {
    'factor_returns': 'factor_returns_full.parquet',
    'regime_probs': 'regime_probabilities.parquet',
    'bl_weights': 'bl_weights_timeseries.parquet',
    'cvar_weights': 'cvar_weights_timeseries.parquet',
    'backtest_returns': 'backtest_returns.parquet',
    'backtest_nav': 'backtest_nav.parquet',
    'garch_vol': 'garch_conditional_vol.parquet',
    'dcc_corr': 'dcc_conditional_corr.parquet',
    'master_data': 'master_data.parquet',
}

for key, filename in parquet_files.items():
    fpath = PROCESSED_DIR / filename
    if fpath.exists():
        data[key] = pd.read_parquet(fpath)
        print(f"  Loaded {filename}: {data[key].shape}")
    else:
        print(f"  MISSING: {filename}")

# Load CSV tables
csv_files = {
    'var_cvar': 'var_cvar_table.csv',
    'evt_params': 'evt_parameters.csv',
    'stress_test': 'stress_test_results.csv',
    'backtest_perf': 'backtest_performance.csv',
}

tables = {}
for key, filename in csv_files.items():
    fpath = TABLES_DIR / filename
    if fpath.exists():
        tables[key] = pd.read_csv(fpath)
        print(f"  Loaded {filename}: {tables[key].shape}")
    else:
        print(f"  MISSING: {filename}")

  Loaded factor_returns_full.parquet: (232, 4)
  Loaded regime_probabilities.parquet: (186, 4)
  Loaded bl_weights_timeseries.parquet: (125, 4)
  Loaded cvar_weights_timeseries.parquet: (185, 4)
  Loaded backtest_returns.parquet: (64, 6)
  Loaded backtest_nav.parquet: (64, 6)
  Loaded garch_conditional_vol.parquet: (232, 4)
  Loaded dcc_conditional_corr.parquet: (231, 6)
  Loaded master_data.parquet: (2556, 27)
  Loaded var_cvar_table.csv: (19, 13)
  Loaded evt_parameters.csv: (19, 8)
  Loaded stress_test_results.csv: (12, 7)
  Loaded backtest_performance.csv: (6, 18)


## 11.2 Excel Dashboard (10 Tabs)

In [3]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows

wb = Workbook()
header_font = Font(bold=True, size=11, color='FFFFFF')
header_fill = PatternFill(start_color='2C3E50', end_color='2C3E50', fill_type='solid')
green_fill = PatternFill(start_color='D5F5E3', end_color='D5F5E3', fill_type='solid')
red_fill = PatternFill(start_color='FADBD8', end_color='FADBD8', fill_type='solid')
thin_border = Border(
    left=Side(style='thin'), right=Side(style='thin'),
    top=Side(style='thin'), bottom=Side(style='thin')
)

def write_df_to_sheet(ws, df, start_row=1, start_col=1):
    """Write DataFrame to worksheet with formatting."""
    # Determine column offset: if index has a name, write it in the first column
    has_idx = df.index.name is not None and df.index.name != ''
    data_col_offset = 1 if has_idx else 0
    
    # Index header
    if has_idx:
        cell = ws.cell(row=start_row, column=start_col, value=str(df.index.name))
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = Alignment(horizontal='center')
        cell.border = thin_border
    
    # Column headers
    for j, col in enumerate(df.columns):
        cell = ws.cell(row=start_row, column=start_col + data_col_offset + j, value=str(col))
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = Alignment(horizontal='center')
        cell.border = thin_border
    
    # Data
    for i, (idx, row) in enumerate(df.iterrows()):
        # Index column
        if has_idx:
            cell = ws.cell(row=start_row + 1 + i, column=start_col, value=str(idx))
            cell.border = thin_border
        
        for j, val in enumerate(row):
            cell = ws.cell(row=start_row + 1 + i, column=start_col + data_col_offset + j)
            if isinstance(val, (int, float, np.integer, np.floating)):
                cell.value = float(val) if np.isfinite(val) else None
                cell.number_format = '0.0000'
                # Conditional formatting
                if isinstance(val, (float, np.floating)) and np.isfinite(val):
                    if 'return' in str(df.columns[j]).lower() or 'sharpe' in str(df.columns[j]).lower():
                        cell.fill = green_fill if val > 0 else red_fill
            else:
                cell.value = str(val) if val is not None else ''
            cell.border = thin_border

print("Building Excel dashboard...")

Building Excel dashboard...


In [4]:
# Tab 1: Executive Summary
ws1 = wb.active
ws1.title = "Executive Summary"
ws1['A1'] = "Cross-Sector Factor Timing & Dynamic Allocation Engine"
ws1['A1'].font = Font(bold=True, size=16)
ws1['A3'] = "Investment Thesis"
ws1['A3'].font = Font(bold=True, size=12)
ws1['A4'] = "Equity factor premia exhibit regime-dependent behaviour. Dynamic re-weighting"
ws1['A5'] = "conditional on detected macroeconomic regimes should deliver superior risk-adjusted returns."
ws1['A7'] = "Key Results"
ws1['A7'].font = Font(bold=True, size=12)

row = 8
if 'backtest_returns' in data:
    bt = data['backtest_returns']
    for col in bt.columns:
        ann_ret = bt[col].mean() * 12
        ann_vol = bt[col].std() * np.sqrt(12)
        sharpe = ann_ret / ann_vol if ann_vol > 0 else 0
        ws1.cell(row=row, column=1, value=col)
        ws1.cell(row=row, column=2, value=f"Return: {ann_ret:.2%}")
        ws1.cell(row=row, column=3, value=f"Vol: {ann_vol:.2%}")
        ws1.cell(row=row, column=4, value=f"Sharpe: {sharpe:.3f}")
        row += 1

print("  Tab 1: Executive Summary")

  Tab 1: Executive Summary


In [5]:
# Tab 2: Performance Summary
ws2 = wb.create_sheet("Performance Summary")
if 'backtest_returns' in data:
    bt = data['backtest_returns']
    perf_data = []
    for col in bt.columns:
        r = bt[col].dropna()
        ann_ret = r.mean() * 12
        ann_vol = r.std() * np.sqrt(12)
        cum = (1 + r).cumprod()
        max_dd = (cum / cum.cummax() - 1).min()
        downside = r[r < 0].std() * np.sqrt(12)
        perf_data.append({
            'Strategy': col,
            'Ann. Return': ann_ret,
            'Ann. Volatility': ann_vol,
            'Sharpe': ann_ret / ann_vol if ann_vol > 0 else 0,
            'Sortino': ann_ret / downside if downside > 0 else 0,
            'Max Drawdown': max_dd,
            'Calmar': ann_ret / abs(max_dd) if max_dd != 0 else 0,
        })
    perf_df = pd.DataFrame(perf_data)
    write_df_to_sheet(ws2, perf_df)
print("  Tab 2: Performance Summary")

# Tab 3: Annual Returns
ws3 = wb.create_sheet("Annual Returns")
if 'backtest_returns' in data:
    annual = data['backtest_returns'].resample('YE').apply(lambda x: (1+x).prod() - 1)
    annual.index = annual.index.year
    write_df_to_sheet(ws3, annual.round(4))
print("  Tab 3: Annual Returns")

  Tab 2: Performance Summary
  Tab 3: Annual Returns


In [6]:
# Tab 4: Regime Timeline
ws4 = wb.create_sheet("Regime Timeline")
if 'regime_probs' in data:
    rp = data['regime_probs'].copy()
    rp.index = rp.index.strftime('%Y-%m')
    write_df_to_sheet(ws4, rp.round(4).tail(60))  # Last 5 years
print("  Tab 4: Regime Timeline")

# Tab 5: Weight History
ws5 = wb.create_sheet("Weight History")
if 'bl_weights' in data:
    wts = data['bl_weights'].copy()
    wts.index = wts.index.strftime('%Y-%m')
    write_df_to_sheet(ws5, wts.round(4).tail(60))
print("  Tab 5: Weight History")

# Tab 6: Stress Test Results
ws6 = wb.create_sheet("Stress Tests")
if 'stress_test' in tables:
    write_df_to_sheet(ws6, tables['stress_test'].round(4))
print("  Tab 6: Stress Tests")

# Tab 7: EVT Tail Metrics
ws7 = wb.create_sheet("EVT Metrics")
if 'evt_params' in tables:
    write_df_to_sheet(ws7, tables['evt_params'].round(4))
print("  Tab 7: EVT Metrics")

  Tab 4: Regime Timeline
  Tab 5: Weight History
  Tab 6: Stress Tests
  Tab 7: EVT Metrics


In [7]:
# Tab 8: Factor Conditional Stats
ws8 = wb.create_sheet("Factor Stats")
if 'factor_returns' in data and 'regime_probs' in data:
    fr = data['factor_returns']
    factors = ['hml', 'umd', 'rmw', 'lowvol']
    avail = [f for f in factors if f in fr.columns]
    stats = fr[avail].describe().T
    stats['ann_return'] = fr[avail].mean() * 12
    stats['ann_vol'] = fr[avail].std() * np.sqrt(12)
    stats['sharpe'] = stats['ann_return'] / stats['ann_vol']
    write_df_to_sheet(ws8, stats.round(4))
print("  Tab 8: Factor Stats")

# Tab 9: Hypothesis Tests
ws9 = wb.create_sheet("Hypothesis Tests")
ws9['A1'] = "Hypothesis Test Results"
ws9['A1'].font = Font(bold=True, size=12)
hypotheses = [
    ("H1", "Momentum crashes in crisis-to-recovery transitions", "See Phase 5 results"),
    ("H2", "Value outperforms in early expansion", "See Phase 5 results"),
    ("H3", "Quality/LowVol dominate in crisis", "See Phase 5 results"),
    ("H4", "Cross-factor correlations increase in crisis", "See Phase 6 results"),
]
for i, (h_id, desc, result) in enumerate(hypotheses):
    ws9.cell(row=3+i, column=1, value=h_id)
    ws9.cell(row=3+i, column=2, value=desc)
    ws9.cell(row=3+i, column=3, value=result)
print("  Tab 9: Hypothesis Tests")

# Tab 10: Model Diagnostics
ws10 = wb.create_sheet("Diagnostics")
ws10['A1'] = "Model Diagnostics Summary"
ws10['A1'].font = Font(bold=True, size=12)
garch_path = TABLES_DIR / 'factor_garch_parameters.csv'
if garch_path.exists():
    garch_df = pd.read_csv(garch_path)
    write_df_to_sheet(ws10, garch_df.round(4), start_row=3)
print("  Tab 10: Diagnostics")

  Tab 8: Factor Stats
  Tab 9: Hypothesis Tests
  Tab 10: Diagnostics


In [8]:
# Save Excel dashboard
excel_path = REPORTS_DIR / 'factor_timing_dashboard.xlsx'
wb.save(str(excel_path))
print(f"\nExcel dashboard saved: {excel_path}")
print(f"Tabs: {[ws.title for ws in wb.worksheets]}")
logger.info(f"Excel dashboard exported: {excel_path}")


Excel dashboard saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/reports/factor_timing_dashboard.xlsx
Tabs: ['Executive Summary', 'Performance Summary', 'Annual Returns', 'Regime Timeline', 'Weight History', 'Stress Tests', 'EVT Metrics', 'Factor Stats', 'Hypothesis Tests', 'Diagnostics']


## 11.3 PDF Report

In [9]:
# Generate PDF report using matplotlib PdfPages
pdf_path = REPORTS_DIR / 'factor_timing_report.pdf'

with PdfPages(str(pdf_path)) as pdf:
    
    # Page 1: Title
    fig, ax = plt.subplots(figsize=(11, 8.5))
    ax.axis('off')
    ax.text(0.5, 0.65, 'Cross-Sector Factor Timing\n& Dynamic Allocation Engine',
            transform=ax.transAxes, fontsize=28, fontweight='bold',
            ha='center', va='center', color='#2C3E50')
    ax.text(0.5, 0.45, 'Walk-Forward Backtest Report',
            transform=ax.transAxes, fontsize=18, ha='center', color='#7F8C8D')
    ax.text(0.5, 0.35, f'Generated: {datetime.now():%Y-%m-%d}',
            transform=ax.transAxes, fontsize=12, ha='center', color='#95A5A6')
    ax.text(0.5, 0.25, 'Backtest Period: 2010-01 to 2025-12 (out-of-sample)',
            transform=ax.transAxes, fontsize=12, ha='center', color='#95A5A6')
    pdf.savefig(fig, dpi=300)
    plt.close()
    
    # Page 2: Executive Summary
    fig, ax = plt.subplots(figsize=(11, 8.5))
    ax.axis('off')
    ax.text(0.05, 0.95, 'Executive Summary', fontsize=20, fontweight='bold',
            transform=ax.transAxes, va='top', color='#2C3E50')
    
    summary_text = (
        "Investment Thesis:\n"
        "Equity factor premia (value, momentum, quality, low-volatility) exhibit\n"
        "regime-dependent behaviour. A systematic strategy that detects regime\n"
        "transitions using observable macro data and dynamically re-weights factor\n"
        "exposures should deliver superior risk-adjusted returns.\n\n"
        "Methodology:\n"
        "- 3-state HMM regime detection on composite macro index (expanding-window)\n"
        "- Black-Litterman allocation with regime-conditional views\n"
        "- Mean-CVaR risk parity with regime tilts\n"
        "- Walk-forward backtest with 25 bps transaction costs\n"
        "- Strict anti-leakage: filtered probabilities, no future information\n\n"
        "Key Findings:\n"
        "- Factor premia vary significantly across macroeconomic regimes\n"
        "- Dynamic allocation can improve risk-adjusted returns\n"
        "- Tail risk (EVT/Copula) confirms heavy tails and joint crash risk"
    )
    ax.text(0.05, 0.85, summary_text, fontsize=11, transform=ax.transAxes,
            va='top', family='monospace', linespacing=1.5)
    pdf.savefig(fig, dpi=300)
    plt.close()
    
    # Page 3: Performance Summary Table
    if 'backtest_returns' in data:
        fig, ax = plt.subplots(figsize=(11, 8.5))
        ax.axis('off')
        ax.text(0.05, 0.95, 'Performance Summary', fontsize=20, fontweight='bold',
                transform=ax.transAxes, va='top', color='#2C3E50')
        
        if len(perf_data) > 0:
            perf_table = pd.DataFrame(perf_data).round(4)
            table = ax.table(cellText=perf_table.values,
                            colLabels=perf_table.columns,
                            cellLoc='center', loc='center')
            table.auto_set_font_size(False)
            table.set_fontsize(9)
            table.scale(1.2, 1.5)
            
            # Header formatting
            for j in range(len(perf_table.columns)):
                table[0, j].set_facecolor('#2C3E50')
                table[0, j].set_text_props(color='white', fontweight='bold')
        
        pdf.savefig(fig, dpi=300)
        plt.close()
    
    # Page 4: NAV Curves
    if 'backtest_nav' in data:
        fig, ax = plt.subplots(figsize=(11, 6))
        data['backtest_nav'].plot(ax=ax, linewidth=1.2)
        ax.set_title('Strategy NAV Comparison (Base = 100)', fontsize=14, fontweight='bold')
        ax.set_ylabel('NAV')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        pdf.savefig(fig, dpi=300)
        plt.close()
    
    # Page 5: Weight Evolution
    if 'bl_weights' in data:
        fig, ax = plt.subplots(figsize=(11, 6))
        colours_list = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6']
        data['bl_weights'].plot.area(ax=ax, color=colours_list, alpha=0.8)
        ax.set_title('Black-Litterman Factor Allocation', fontsize=14, fontweight='bold')
        ax.set_ylabel('Weight')
        ax.set_ylim(0, 1.05)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        pdf.savefig(fig, dpi=300)
        plt.close()
    
    # Page 6: Regime Timeline
    if 'regime_probs' in data:
        prob_cols = [c for c in data['regime_probs'].columns if c.startswith('p_')]
        if prob_cols:
            fig, ax = plt.subplots(figsize=(11, 5))
            data['regime_probs'][prob_cols].plot.area(ax=ax, alpha=0.8,
                color=['#2ecc71', '#f39c12', '#e74c3c'][:len(prob_cols)])
            ax.set_title('HMM Regime Probabilities Over Time', fontsize=14, fontweight='bold')
            ax.set_ylabel('Probability')
            ax.set_ylim(0, 1)
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            pdf.savefig(fig, dpi=300)
            plt.close()
    
    # Page 7: Limitations
    fig, ax = plt.subplots(figsize=(11, 8.5))
    ax.axis('off')
    ax.text(0.05, 0.95, 'Limitations & Caveats', fontsize=20, fontweight='bold',
            transform=ax.transAxes, va='top', color='#2C3E50')
    limitations = (
        "1. Survivorship bias: S&P 500 constituent list is current, not historical\n"
        "2. Transaction costs assumed at 25 bps — may be optimistic for less liquid factors\n"
        "3. HMM regime detection is inherently backward-looking; regime transitions\n"
        "   are identified with a lag\n"
        "4. Walk-forward period (2010-2025) includes unusual monetary policy regimes\n"
        "5. Factor returns from French Library are based on full CRSP universe,\n"
        "   not tradeable instruments\n"
        "6. Low-volatility factor construction uses S&P 500 only (survivorship bias)\n"
        "7. Monte Carlo assumes multivariate normal — fails to capture tail events\n"
        "8. Sentiment proxy is market-based, not actual NLP from headlines"
    )
    ax.text(0.05, 0.85, limitations, fontsize=11, transform=ax.transAxes,
            va='top', linespacing=1.8)
    pdf.savefig(fig, dpi=300)
    plt.close()

print(f"PDF report saved: {pdf_path}")
logger.info(f"PDF report exported: {pdf_path}")

PDF report saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/reports/factor_timing_report.pdf


## 11.4 Final Validation

In [10]:
print("=" * 60)
print("PHASE 11 — DELIVERABLES CHECK")
print("=" * 60)

deliverables = {
    'Excel dashboard': REPORTS_DIR / 'factor_timing_dashboard.xlsx',
    'PDF report': REPORTS_DIR / 'factor_timing_report.pdf',
}

for name, path in deliverables.items():
    exists = path.exists()
    size = path.stat().st_size if exists else 0
    print(f"  [{'OK' if exists else 'MISSING'}] {name}: {size:,} bytes")

# Check all parquet outputs
print(f"\nParquet outputs in {PROCESSED_DIR}:")
for f in sorted(PROCESSED_DIR.glob('*.parquet')):
    print(f"  {f.name}: {f.stat().st_size:,} bytes")

print(f"\nCSV tables in {TABLES_DIR}:")
for f in sorted(TABLES_DIR.glob('*.csv')):
    print(f"  {f.name}: {f.stat().st_size:,} bytes")

print(f"\nFigures in {FIGURES_DIR}:")
for f in sorted(FIGURES_DIR.glob('*.png')):
    print(f"  {f.name}: {f.stat().st_size:,} bytes")

logger.info("Phase 11 complete")
print("\n=== Phase 11 Complete ===")

PHASE 11 — DELIVERABLES CHECK
  [OK] Excel dashboard: 16,717 bytes
  [OK] PDF report: 74,215 bytes

Parquet outputs in /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/data/processed:
  backtest_nav.parquet: 8,908 bytes
  backtest_returns.parquet: 8,902 bytes
  bl_weights_timeseries.parquet: 9,263 bytes
  conditional_covariance.parquet: 45,882 bytes
  conditional_vol_series.parquet: 466,264 bytes
  cvar_weights_timeseries.parquet: 9,552 bytes
  dcc_conditional_corr.parquet: 19,769 bytes
  factor_returns.parquet: 17,039 bytes
  factor_returns_full.parquet: 12,536 bytes
  garch_conditional_vol.parquet: 14,239 bytes
  lowvol_factor_returns.parquet: 6,849 bytes
  macro_composite_index.parquet: 5,481 bytes
  macro_indicators.parquet: 26,996 bytes
  macro_regimes.parquet: 26,001 bytes
  master_data.parquet: 474,964 bytes
  regime_labels.parquet: 30,761 bytes
  regime_probabilities.parquet: 10,583 bytes
  sp500_daily_prices.parquet: 16,478,855 bytes
  sp500_monthly